# Distributed Training & Hyperparameter Tuning Demo

## Context & Requirements

**Dataset Profile:**
- ~147 million rows, ~4,200 columns
- 2 columns contain JSON objects with up to 2,000 keys (avg ~50-100 keys)
- Tabular classification/regression workload

**Likely Model Types:**
- **XGBoost** (gradient boosted trees) - best for wide tabular data with sparse features
- **LightGBM** - fast histogram-based gradient boosting, handles JSON-exploded sparse features well
- Both support distributed multi-node training natively

**Deployment Strategy:**
- **Interactive Dev/Demo:** Notebook Container Runtime + `scale_cluster()` for rapid iteration
- **Production:** Multi-Node ML Jobs with `target_instances` for scheduled, reproducible training

**Key Challenges Addressed:**
1. Multi-node scaling: `scale_cluster()` in notebooks, `target_instances` in ML Jobs
2. Distributed training via `XGBEstimator` / `LGBMEstimator` with `feature_cols`
3. HPO on Container Runtime (each trial uses full distributed cluster)
4. Production scheduling via **ML Jobs** or **SPCS Job Service via Task**

---

In [ ]:
import snowflake.snowpark as snowpark
from snowflake.snowpark import Session
import snowflake.snowpark.functions as F
import snowflake.snowpark.types as T
from snowflake.ml.modeling.distributors.xgboost import XGBEstimator
from snowflake.ml.modeling.distributors.lightgbm import LightGBMEstimator
from snowflake.ml.registry import Registry
import numpy as np
import json

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
print(f"Session established: {session.get_current_role()}, WH: {session.get_current_warehouse()}")

In [ ]:
DB = "RRD_ML_DEMO"
SCHEMA = "DISTRIBUTED_TRAINING"
TABLE_NAME = "SYNTHETIC_RRD_DATA"

session.sql(f"CREATE DATABASE IF NOT EXISTS {DB}").collect()
session.sql(f"CREATE SCHEMA IF NOT EXISTS {DB}.{SCHEMA}").collect()
session.sql(f"USE DATABASE {DB}").collect()
session.sql(f"USE SCHEMA {SCHEMA}").collect()
print(f"Using {DB}.{SCHEMA}")

## 1. Synthetic Data Generation

Simulates RRD's dataset structure:
- Scaled down to **1M rows** for demo (production: 147M)
- **50 numeric feature columns** (representative of the 4,200 columns)
- **2 JSON columns** with variable-length key-value objects (up to 100 keys)
- **1 binary target column** (classification task)

> For production at 147M rows x 4,200 cols, use a multi-node warehouse (4XL+) for data prep.

In [ ]:
NUM_ROWS = 1_000_000
NUM_FEATURE_COLS = 50
MAX_JSON_KEYS = 100
AVG_JSON_KEYS = 50

In [ ]:


numeric_cols_ddl = ", ".join([f"FEAT_{i} FLOAT" for i in range(NUM_FEATURE_COLS)])

create_table_sql = f"""
CREATE OR REPLACE TABLE {TABLE_NAME} (
    ID NUMBER AUTOINCREMENT,
    {numeric_cols_ddl},
    JSON_COL_A VARIANT,
    JSON_COL_B VARIANT,
    TARGET NUMBER(1,0)
)
"""
session.sql(create_table_sql).collect()
print(f"Table {TABLE_NAME} created with {NUM_FEATURE_COLS} numeric + 2 JSON + 1 target columns.")

In [ ]:
num_batches = 10
batch_size = NUM_ROWS // num_batches

for batch_idx in range(num_batches):
    feat_selects = ", ".join([
        f"UNIFORM(0::FLOAT, 1::FLOAT, RANDOM({i * 1000 + batch_idx})) AS FEAT_{i}"
        for i in range(NUM_FEATURE_COLS)
    ])

    json_a_keys = f"""OBJECT_CONSTRUCT_KEEP_NULL(
        {', '.join([f"'key_{k}', IFF(UNIFORM(0,1,RANDOM({k})) > 0.5, UNIFORM(0::FLOAT, 100::FLOAT, RANDOM({k}+1)), NULL)" for k in range(AVG_JSON_KEYS)])}
    )"""

    json_b_keys = f"""OBJECT_CONSTRUCT_KEEP_NULL(
        {', '.join([f"'attr_{k}', IFF(UNIFORM(0,1,RANDOM({k}+500)) > 0.4, UNIFORM(-50::FLOAT, 50::FLOAT, RANDOM({k}+501)), NULL)" for k in range(AVG_JSON_KEYS)])}
    )"""

    insert_sql = f"""
    INSERT INTO {TABLE_NAME} ({', '.join([f'FEAT_{i}' for i in range(NUM_FEATURE_COLS)])}, JSON_COL_A, JSON_COL_B, TARGET)
    SELECT
        {feat_selects},
        {json_a_keys} AS JSON_COL_A,
        {json_b_keys} AS JSON_COL_B,
        IFF(UNIFORM(0::FLOAT, 1::FLOAT, RANDOM(999)) > 0.5, 1, 0) AS TARGET
    FROM TABLE(GENERATOR(ROWCOUNT => {batch_size}))
    """
    session.sql(insert_sql).collect()
    print(f"  Batch {batch_idx + 1}/{num_batches} inserted ({batch_size} rows)")

print(f"\nTotal rows inserted: {NUM_ROWS:,}")

In [ ]:
json_keys_a = [f"key_{k}" for k in range(AVG_JSON_KEYS)]
json_keys_b = [f"attr_{k}" for k in range(AVG_JSON_KEYS)]

flatten_a_cols = ", ".join([
    f"JSON_COL_A:{k}::FLOAT AS JSON_A_{k.upper()}" for k in json_keys_a
])
flatten_b_cols = ", ".join([
    f"JSON_COL_B:{k}::FLOAT AS JSON_B_{k.upper()}" for k in json_keys_b
])

feat_cols_select = ", ".join([f"FEAT_{i}" for i in range(NUM_FEATURE_COLS)])

flattened_view_sql = f"""
CREATE OR REPLACE VIEW {TABLE_NAME}_FLAT AS
SELECT
    ID,
    {feat_cols_select},
    {flatten_a_cols},
    {flatten_b_cols},
    TARGET
FROM {TABLE_NAME}
"""
session.sql(flattened_view_sql).collect()
print(f"Created flattened view with {NUM_FEATURE_COLS + AVG_JSON_KEYS * 2} feature columns + TARGET")

In [ ]:
df = session.table(f"{TABLE_NAME}_FLAT")

feature_cols = [f"FEAT_{i}" for i in range(NUM_FEATURE_COLS)] + \
              [f"JSON_A_{k.upper()}" for k in json_keys_a] + \
              [f"JSON_B_{k.upper()}" for k in json_keys_b]

label_col = "TARGET"

df_filled = df.fillna(0.0, subset=feature_cols)

train_df, test_df = df_filled.random_split([0.8, 0.2], seed=42)

# Materialize into tables to avoid re-executing the query during distributed training
train_df.write.mode("overwrite").save_as_table(f"{TABLE_NAME}_TRAIN")
test_df.write.mode("overwrite").save_as_table(f"{TABLE_NAME}_TEST")

train_df = session.table(f"{TABLE_NAME}_TRAIN")
test_df = session.table(f"{TABLE_NAME}_TEST")

print(f"Feature columns: {len(feature_cols)}")
print(f"Training rows: {train_df.count():,}")
print(f"Test rows: {test_df.count():,}")
print(f"Data materialized into {TABLE_NAME}_TRAIN and {TABLE_NAME}_TEST")

## 2. Multi-Node Cluster Scaling (Notebook - Interactive Demo)

**For Notebooks (interactive dev/demo):** Call `scale_cluster()` to activate multiple nodes.
Without this call, even if `max_nodes > 1` on the compute pool, the runtime only uses a single node.

```python
from snowflake.ml.runtime_cluster import scale_cluster
scale_cluster(expected_cluster_size=4)  # 1 head + 3 workers
```

**For ML Jobs (production):** Use `target_instances=N` in the `@remote` decorator or `submit_file()`.
No `scale_cluster()` call needed — the job automatically provisions N nodes.

> This notebook demonstrates the **Notebook approach** for interactive exploration.
> Section 8 shows the **ML Jobs approach** for production deployment.

In [ ]:
from snowflake.ml.runtime_cluster import scale_cluster, get_nodes

NUM_NODES = 4
gpu_workload = False

scale_cluster(expected_cluster_size=NUM_NODES)
nodes = get_nodes()
print(f"Cluster scaled to {NUM_NODES} total nodes (1 head + {NUM_NODES - 1} workers).")
print(f"Active nodes: {len(nodes)}")
for node in nodes:
    print(f"  {node['name']} - CPUs: {node['cpus']}, GPUs: {node['gpus']}")

## 3. Distributed XGBoost Training

Using `snowflake.ml.modeling.distributors.xgboost.XGBEstimator` which:
- Distributes data across cluster nodes automatically
- Uses `feature_cols` and `label_col` parameters in `fit()` (NOT passed to constructor)
- Runs distributed XGBoost via Ray backend on container runtime

> **Note on Tuner limitation:** The Tuner object cannot directly pass `feature_cols` to distributed estimators' `fit()` method.
> Workaround: Use a custom `train_func` with the Tuner (shown in Section 5).

In [ ]:
import time
from snowflake.ml.modeling.distributors.xgboost import XGBEstimator, XGBScalingConfig
from snowflake.ml.data.data_connector import DataConnector

# Load pre-materialized training data
train_df = session.table(f"{DB}.{SCHEMA}.{TABLE_NAME}_TRAIN")

# Derive feature columns from table schema (all columns except ID and TARGET)
feature_cols = [c for c in train_df.columns if c not in ("ID", "TARGET")]
label_col = "TARGET"

xgb_params = {
    "max_depth": 8,
    "learning_rate": 0.1,
    "subsample": 0.8,
    "colsample_bytree": 0.6,
    "tree_method": "hist",
    "objective": "binary:logistic",
    "eval_metric": "auc",
}

scaling_config = XGBScalingConfig(num_workers=4,
                                  num_cpu_per_worker=28, 
                                  use_gpu=gpu_workload)

xgb_estimator = XGBEstimator(
    n_estimators=100,
    params=xgb_params,
    scaling_config=scaling_config,
)

# DataConnector from pre-materialized table — no query execution during fit()
train_connector = DataConnector.from_dataframe(train_df)

print("Training Distributed XGBoost...")
print(f"  Nodes: {NUM_NODES}")
print(f"  Workers: 4, CPUs per worker: 24")
print(f"  Training rows: {train_df.count():,}")
print(f"  Features: {len(feature_cols)}")

xgb_start_time = time.time()
xgb_model = xgb_estimator.fit(
    train_connector,
    input_cols=feature_cols,
    label_col=label_col
)
xgb_train_duration = time.time() - xgb_start_time

print(f"\nXGBoost training complete (distributed across {NUM_NODES} nodes).")
print(f"Training time: {xgb_train_duration:.1f} seconds ({xgb_train_duration/60:.1f} min)")

In [ ]:
import pandas as pd
import xgboost
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, precision_score, recall_score

# Load pre-materialized test data
test_df = session.table(f"{DB}.{SCHEMA}.{TABLE_NAME}_TEST")

print("Running single-node prediction...")
xgb_pred_start = time.time()

# Single-node prediction — faster for small datasets, avoids Ray coordination overhead
test_pd = test_df.to_pandas()
dtest = xgboost.DMatrix(test_pd[feature_cols])
y_prob = xgb_model.predict(dtest)

xgb_pred_duration = time.time() - xgb_pred_start

y_true = test_pd[label_col].astype(int)
y_pred = (y_prob >= 0.5).astype(int)

train_count = train_df.count()

xgb_accuracy = accuracy_score(y_true, y_pred)
xgb_auc = roc_auc_score(y_true, y_prob)
xgb_f1 = f1_score(y_true, y_pred)
xgb_precision = precision_score(y_true, y_pred)
xgb_recall = recall_score(y_true, y_pred)

print(f"\n{'='*60}")
print(f"DISTRIBUTED XGBoost PERFORMANCE METRICS")
print(f"{'='*60}")
print(f"")
print(f"--- Training Metrics (Demo Run) ---")
print(f"  Training time:       {xgb_train_duration:.1f}s ({xgb_train_duration/60:.1f} min)")
print(f"  Prediction time:     {xgb_pred_duration:.1f}s")
print(f"  Nodes used:          {NUM_NODES}")
print(f"  Throughput (train):  {train_count / xgb_train_duration:,.0f} rows/sec")
print(f"  Features:            {len(feature_cols)}")
print(f"  Training rows:       {train_count:,}")
print(f"")
print(f"--- Model Quality (random synthetic data, expect ~0.5) ---")
print(f"  Accuracy:            {xgb_accuracy:.4f}")
print(f"  AUC-ROC:             {xgb_auc:.4f}")
print(f"  F1 Score:            {xgb_f1:.4f}")
print(f"  Precision:           {xgb_precision:.4f}")
print(f"  Recall:              {xgb_recall:.4f}")
print(f"{'='*60}")

## Compute Pool Sizing Recommendations for RR Donnelley

### Dataset: 147M rows x 4,200 columns (+ JSON expansion ~ 4,400 effective features)

| Instance Family | vCPU | Memory | GPU | Best For |
|----------------|------|--------|-----|----------|
| `CPU_X64_M` | 8 | 64 GB | - | Data prep, feature engineering |
| `CPU_X64_L` | 32 | 256 GB | - | Large-scale CPU-based XGBoost |
| `HIGHMEM_X64_M` | 8 | 128 GB | - | Wide datasets needing more memory per worker |
| `GPU_NV_S` | 8 | 64 GB | 1x A10G (24GB) | Single-GPU training, small HPO |
| `GPU_NV_M` | 48 | 384 GB | 4x A10G (96GB) | **Recommended for RRD** - multi-GPU per node |
| `GPU_NV_L` | 96 | 1.1 TB | 8x A100 (640GB) | Maximum throughput, large-scale HPO |

### Recommendation for RRD (147M rows x 4,400 features):

**Development/Experimentation:**
```sql
CREATE COMPUTE POOL RRD_DEV_POOL
  MIN_NODES = 1
  MAX_NODES = 4
  INSTANCE_FAMILY = GPU_NV_M
  AUTO_SUSPEND_SECS = 900;
```
- 4 nodes x 4 GPUs = 16 GPUs total
- ~384 GB GPU memory for model training

**Production Training:**
```sql
CREATE COMPUTE POOL RRD_PROD_POOL
  MIN_NODES = 4
  MAX_NODES = 8
  INSTANCE_FAMILY = GPU_NV_M
  AUTO_SUSPEND_SECS = 300;
```
- 8 nodes x 4 GPUs = 32 GPUs
- Handles 147M rows with feature distribution across workers
- Auto-suspend after 5 min when idle

**Why GPU_NV_M (not GPU_NV_L)?**
- XGBoost/LightGBM on tabular data don't need A100-class GPUs
- A10G has sufficient memory (24GB each, 96GB per node) for tree-based models
- 4x A10G per node is better matched to tree-based workloads than 8x A100
- RRD's data is tabular (not deep learning), so A10G throughput is sufficient

In [ ]:
import pandas as pd
import math

train_count = train_df.count()

PROD_ROWS = 147_000_000
PROD_FEATURES = 4_400
DEMO_ROWS = train_count
DEMO_FEATURES = len(feature_cols)
DEMO_NODES = NUM_NODES

overhead_secs = 14.0
actual_compute_time = xgb_train_duration - overhead_secs

row_ratio = PROD_ROWS / DEMO_ROWS
feat_ratio = PROD_FEATURES / DEMO_FEATURES

row_factor = row_ratio
feat_factor = math.log2(PROD_FEATURES) / math.log2(DEMO_FEATURES)

base_time_same_nodes = actual_compute_time * row_factor * feat_factor

eff_8 = 0.85
eff_16 = 0.70

est_4 = base_time_same_nodes
est_8 = base_time_same_nodes * (DEMO_NODES / 8) / eff_8
est_16 = base_time_same_nodes * (DEMO_NODES / 16) / eff_16

colsample_savings = 0.4
est_4_opt = est_4 * colsample_savings
est_8_opt = est_8 * colsample_savings
est_16_opt = est_16 * colsample_savings

print(f"{'='*70}")
print(f"PRODUCTION EXTRAPOLATION")
print(f"{'='*70}")
print(f"")
print(f"--- Demo Baseline (measured) ---")
print(f"  Rows:             {DEMO_ROWS:,}")
print(f"  Features:         {DEMO_FEATURES}")
print(f"  Compute time:     {actual_compute_time:.0f}s ({actual_compute_time/60:.1f} min)")
print(f"  Nodes:            {DEMO_NODES}")
print(f"  n_estimators:     200")
print(f"")
print(f"--- Production Target ---")
print(f"  Rows:             {PROD_ROWS:,} ({row_ratio:.0f}x)")
print(f"  Features:         {PROD_FEATURES} ({feat_ratio:.1f}x)")
print(f"")
print(f"--- Scaling Model ---")
print(f"  XGBoost 'hist' uses fixed 256 bins per feature - scales linearly with rows,")
print(f"  sub-linearly with features (log ratio: {feat_factor:.2f}x).")
print(f"  Distributed XGBoost partitions rows across workers (near-linear node scaling).")
print(f"  Communication overhead: ~15% at 2x nodes, ~30% at 4x nodes.")
print(f"")
print(f"--- Estimated Training Time (200 trees, GPU_NV_M) ---")
print(f"  {'Config':<24} {'Raw Estimate':<18} {'With colsample=0.3'}")
for label, est_raw, est_opt in [
    ("4 nodes (16 GPUs)", est_4, est_4_opt),
    ("8 nodes (32 GPUs)", est_8, est_8_opt),
    ("16 nodes (64 GPUs)", est_16, est_16_opt),
]:
    print(f"  {label:<24} {est_raw/3600:>5.1f} hrs          {est_opt/3600:>5.1f} hrs")
print(f"")
print(f"  colsample_bytree=0.3 means each tree uses only 1,320 of 4,400 features")
print(f"  This reduces computation by ~60% with minimal quality loss on wide data.")
print(f"")
print(f"--- Recommendation for RRD ---")
print(f"  POOL:      GPU_NV_M, 8 nodes (32 A10G GPUs, 768 GB GPU memory)")
print(f"  PARAMS:    colsample_bytree=0.3, max_depth=8, tree_method='hist'")
print(f"  STRATEGY:  Benchmark on 10M row sample first, then scale to full 147M")
print(f"  SCHEDULE:  Weekly retrain via ML Job (target_instances=8)")
print(f"{'='*70}")

## 4. Distributed LightGBM Training

LightGBM is often faster than XGBoost for high-dimensional sparse data (like RRD's JSON-exploded features).
Same distributed API pattern as XGBoost.

In [ ]:
lgbm_params = {
    "n_estimators": 200,
    "max_depth": 10,
    "learning_rate": 0.05,
    "num_leaves": 127,
    "subsample": 0.8,
    "colsample_bytree": 0.6,
    "objective": "binary",
    "metric": "auc",
    "n_jobs": -1,
    "verbose": -1,
}

lgbm_estimator = LGBMEstimator(**lgbm_params)

print("Training Distributed LightGBM...")
lgbm_estimator.fit(
    train_df,
    feature_cols=feature_cols,
    label_col=label_col
)
print("LightGBM training complete (distributed across cluster).")

In [ ]:
lgbm_predictions = lgbm_estimator.predict(test_df)

lgbm_pred_df = lgbm_predictions.select(
    F.col("TARGET"),
    F.col("PREDICTION")
).to_pandas()

lgbm_accuracy = accuracy_score(lgbm_pred_df["TARGET"], lgbm_pred_df["PREDICTION"])
lgbm_auc = roc_auc_score(lgbm_pred_df["TARGET"], lgbm_pred_df["PREDICTION"])

print(f"LightGBM Results:")
print(f"  Accuracy: {lgbm_accuracy:.4f}")
print(f"  AUC-ROC:  {lgbm_auc:.4f}")
print(classification_report(lgbm_pred_df["TARGET"], lgbm_pred_df["PREDICTION"]))

## 5. Hyperparameter Tuning with Snowflake ML Tuner API

Using the Snowflake ML HPO (Hyperparameter Optimization) API with the `Tuner` class.
The Tuner distributes trials across available resources and supports Bayesian optimization,
random search, and grid search. Each trial's `train_func` uses the distributed `XGBEstimator`
which leverages the full multi-node cluster.

Key components:
- `Tuner`: Orchestrates the HPO process
- `TunerConfig`: Configures metric, mode, search algorithm, and concurrency
- `get_tuner_context()`: Called inside `train_func` to get hyperparams and report metrics
- `DataConnector`: Passes data efficiently to each trial

In [ ]:
from snowflake.ml.modeling import tune
from snowflake.ml.modeling.tune import Tuner, TunerConfig, get_tuner_context, uniform, choice, randint
from snowflake.ml.modeling.tune.search import RandomSearch
from snowflake.ml.modeling.distributors.xgboost import XGBEstimator, XGBScalingConfig
from snowflake.ml.data.data_connector import DataConnector
from sklearn.metrics import roc_auc_score

search_space = {
    "n_estimators": choice([100, 200, 300]),
    "max_depth": randint(6, 12),
    "learning_rate": uniform(0.01, 0.1),
    "subsample": uniform(0.7, 0.9),
    "colsample_bytree": uniform(0.5, 0.9),
}

tuner_config = TunerConfig(
    metric="auc",
    mode="max",
    search_alg=RandomSearch(random_state=42),
    num_trials=2,
    max_concurrent_trials=1,
)

print(f"Search space defined with {len(search_space)} hyperparameters")
print(f"Tuner config: {tuner_config.num_trials} trials, metric={tuner_config.metric}, mode={tuner_config.mode}")

In [ ]:
train_connector = DataConnector.from_dataframe(train_df)
test_connector = DataConnector.from_dataframe(test_df)

dataset_map = {
    "train": train_connector,
    "test": test_connector,
}

# Capture outer-scope variables so they survive serialization into Tuner workers
_feature_cols = feature_cols
_label_col = label_col

def train_func():
    from snowflake.ml.modeling.tune import get_tuner_context
    from snowflake.ml.modeling.distributors.xgboost import XGBEstimator, XGBScalingConfig
    from sklearn.metrics import roc_auc_score
    import pandas as pd

    tuner_context = get_tuner_context()
    config = tuner_context.get_hyper_params()
    dm = tuner_context.get_dataset_map()

    params = {
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "tree_method": "hist",
        "max_depth": int(config["max_depth"]),
        "learning_rate": config["learning_rate"],
        "subsample": config["subsample"],
        "colsample_bytree": config["colsample_bytree"],
    }

    scaling_config = XGBScalingConfig()

    estimator = XGBEstimator(
        n_estimators=int(config["n_estimators"]),
        params=params,
        scaling_config=scaling_config,
    )

    booster = estimator.fit(
        dm["train"],
        input_cols=_feature_cols,
        label_col=_label_col,
    )

    predictions = estimator.predict(dm["test"])
    pred_df = predictions if isinstance(predictions, pd.DataFrame) else predictions.to_pandas()
    pred_col = [c for c in pred_df.columns if "predict" in c.lower()][0]

    # Handle potential case mismatch in column names
    if _label_col in pred_df.columns:
        target_col = _label_col
    elif _label_col.upper() in pred_df.columns:
        target_col = _label_col.upper()
    else:
        target_col = _label_col

    y_true = pred_df[target_col].astype(int)
    y_prob = pred_df[pred_col]
    auc = roc_auc_score(y_true, y_prob)

    tuner_context.report(metrics={"auc": auc}, model=booster)

tuner = Tuner(train_func, search_space, tuner_config)

print("Starting Tuner HPO (each trial uses distributed XGBEstimator across full cluster)...")
tuner_results = tuner.run(dataset_map=dataset_map)
print("\nTuner HPO Complete.")

In [ ]:
import pandas as pd

print("HPO Results (all trials):")
print(tuner_results.results.sort_values("auc", ascending=False).to_string(index=False))

print(f"\nBest Trial:")
print(tuner_results.best_result)

if tuner_results.best_model is not None:
    print(f"\nBest model available for registration.")
else:
    print(f"\nNo best model returned (retrain with best params in next cell).")

## 6. Register Best Model

In [ ]:
best_params_dict = {
    "n_estimators": int(best_trial["n_estimators"]),
    "max_depth": int(best_trial["max_depth"]),
    "learning_rate": best_trial["learning_rate"],
    "subsample": best_trial["subsample"],
    "colsample_bytree": best_trial["colsample_bytree"],
    "tree_method": "hist",
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "n_jobs": -1,
}

print("Retraining best model with full training set...")
best_xgb = XGBEstimator(**best_params_dict)
best_xgb.fit(train_df, feature_cols=feature_cols, label_col=label_col)

final_preds = best_xgb.predict(test_df)
final_pdf = final_preds.select(F.col("TARGET"), F.col("PREDICTION")).to_pandas()
final_auc = roc_auc_score(final_pdf["TARGET"], final_pdf["PREDICTION"])
final_acc = accuracy_score(final_pdf["TARGET"], final_pdf["PREDICTION"])

print(f"Final Model - AUC: {final_auc:.4f}, Accuracy: {final_acc:.4f}")

In [ ]:
registry = Registry(session=session, database_name=DB, schema_name=SCHEMA)

model_version = registry.log_model(
    model=best_xgb,
    model_name="RRD_XGB_DISTRIBUTED",
    version_name="v1",
    sample_input_data=train_df.select(feature_cols).limit(100),
    metrics={
        "auc_roc": final_auc,
        "accuracy": final_acc,
        "num_features": len(feature_cols),
        "training_rows": NUM_ROWS,
        "num_hpo_trials": len(sampled_combos),
    },
    comment="RRD distributed XGBoost - HPO optimized on multi-node SPCS cluster"
)

print(f"Model registered: RRD_XGB_DISTRIBUTED v1")
print(f"Registry: {DB}.{SCHEMA}")

## 7. Production HPO via Multi-Node ML Jobs

**ML Jobs** are the recommended production approach for scheduled HPO. Key advantages:
- `target_instances=N` provisions multi-node clusters automatically (no `scale_cluster()` needed)
- Full Tuner API works inside ML Jobs (distributed HPO with distributed training per trial)
- Integrates natively with **Snowflake Task Graphs** via `DAGTask(definition=...)`
- No container image builds, no YAML specs, no `EXECUTE JOB SERVICE`

| | Old: SPCS Job Service | New: ML Jobs |
|---|---|---|
| **Setup** | Build image, write YAML spec, upload to stage | Just define a `@remote` function |
| **Multi-node** | Manual in container spec | `target_instances=N` |
| **HPO** | Manual loop | Full Tuner API |
| **Scheduling** | SQL Task + EXECUTE JOB SERVICE | DAGTask with job definition |
| **Monitoring** | SYSTEM$GET_JOB_STATUS | `job.status`, `job.get_logs()`, Ray Dashboard |

In [ ]:
from snowflake.ml.jobs import remote

COMPUTE_POOL = "HPO_POOL"
TARGET_NODES = 3

@remote(
    COMPUTE_POOL,
    stage_name="RRD_ML_DEMO.DISTRIBUTED_TRAINING.HPO_STAGE",
    session=session,
    target_instances=TARGET_NODES,
)
def hpo_distributed_job(table_fqn: str, num_trials: int = 12):
    from snowflake.ml.modeling.tune import Tuner, TunerConfig, get_tuner_context, uniform, choice, randint
    from snowflake.ml.modeling.tune.search import RandomSearch
    from snowflake.ml.modeling.distributors.xgboost import XGBEstimator, XGBScalingConfig
    from snowflake.ml.data.data_connector import DataConnector
    from snowflake.ml.registry import Registry
    from snowflake.snowpark.context import get_active_session
    from sklearn.metrics import roc_auc_score
    import pandas as pd

    session = get_active_session()

    df = session.table(table_fqn).fillna(0.0)
    feature_cols = [c for c in df.columns if c not in ("TARGET", "ID")]
    label_col = "TARGET"
    train_df, test_df = df.random_split([0.8, 0.2], seed=42)

    train_connector = DataConnector.from_dataframe(train_df)
    test_connector = DataConnector.from_dataframe(test_df)
    dataset_map = {"train": train_connector, "test": test_connector}

    search_space = {
        "n_estimators": choice([100, 200, 300, 400]),
        "max_depth": randint(6, 12),
        "learning_rate": uniform(0.01, 0.1),
        "subsample": uniform(0.7, 0.9),
        "colsample_bytree": uniform(0.5, 0.9),
    }

    tuner_config = TunerConfig(
        metric="auc",
        mode="max",
        search_alg=RandomSearch(random_state=42),
        num_trials=num_trials,
        max_concurrent_trials=1,
    )

    # Capture variables explicitly for serialization into Tuner worker processes
    _feature_cols = feature_cols
    _label_col = label_col

    def train_func():
        from snowflake.ml.modeling.tune import get_tuner_context
        from snowflake.ml.modeling.distributors.xgboost import XGBEstimator, XGBScalingConfig
        from sklearn.metrics import roc_auc_score
        import pandas as pd

        tuner_context = get_tuner_context()
        config = tuner_context.get_hyper_params()
        dm = tuner_context.get_dataset_map()

        params = {
            "objective": "binary:logistic",
            "eval_metric": "auc",
            "tree_method": "hist",
            "max_depth": int(config["max_depth"]),
            "learning_rate": config["learning_rate"],
            "subsample": config["subsample"],
            "colsample_bytree": config["colsample_bytree"],
        }

        scaling_config = XGBScalingConfig()
        estimator = XGBEstimator(
            n_estimators=int(config["n_estimators"]),
            params=params,
            scaling_config=scaling_config,
        )

        booster = estimator.fit(dm["train"], input_cols=_feature_cols, label_col=_label_col)

        predictions = estimator.predict(dm["test"])
        pred_df = predictions if isinstance(predictions, pd.DataFrame) else predictions.to_pandas()
        pred_col = [c for c in pred_df.columns if "predict" in c.lower()][0]

        # Handle potential case mismatch in column names from distributed estimator
        if _label_col in pred_df.columns:
            target_col = _label_col
        elif _label_col.upper() in pred_df.columns:
            target_col = _label_col.upper()
        else:
            target_col = _label_col

        y_true = pred_df[target_col].astype(int)
        y_prob = pred_df[pred_col]
        auc = roc_auc_score(y_true, y_prob)

        tuner_context.report(metrics={"auc": auc}, model=booster)

    tuner = Tuner(train_func, search_space, tuner_config)
    tuner_results = tuner.run(dataset_map=dataset_map)

    best_auc = float(tuner_results.best_result["auc"].iloc[0])
    registry = Registry(session=session, database_name="RRD_ML_DEMO", schema_name="DISTRIBUTED_TRAINING")
    if tuner_results.best_model is not None:
        registry.log_model(
            model=tuner_results.best_model,
            model_name="RRD_XGB_HPO_PROD",
            version_name="latest",
            metrics={"auc_roc": best_auc, "num_trials": num_trials, "nodes_used": TARGET_NODES},
            comment=f"HPO best model - {num_trials} trials on {TARGET_NODES}-node ML Job"
        )

    return {"best_auc": best_auc, "num_trials": num_trials, "nodes": TARGET_NODES}

print(f"ML Job HPO function defined: hpo_distributed_job")
print(f"  Compute Pool: {COMPUTE_POOL}")
print(f"  Target Instances: {TARGET_NODES} (auto-provisioned, no scale_cluster needed)")
print(f"  HPO: Full Tuner API with distributed XGBEstimator per trial")

In [ ]:
print("Submitting HPO ML Job...")
job = hpo_distributed_job(
    table_fqn=f"{DB}.{SCHEMA}.{TABLE_NAME}_FLAT",
    num_trials=12,
)

print(f"Job submitted: {job.id}")
print(f"Status: {job.status}")
print(f"")
print("Monitor with:")
print(f"  job.status              -> current status")
print(f"  job.wait()              -> block until complete")
print(f"  job.result()            -> get return value (best_auc, etc.)")
print(f"  job.get_logs()          -> head node logs")
print(f"  job.get_logs(instance_id=1) -> worker node logs")
print(f"  job.get_ray_dashboard_url() -> Ray Dashboard for monitoring")

In [ ]:
job.wait()
result = job.result()
print(f"HPO Job completed with status: {job.status}")
print(f"")
print(f"Results:")
print(f"  Best AUC:    {result['best_auc']:.4f}")
print(f"  Trials run:  {result['num_trials']}")
print(f"  Nodes used:  {result['nodes']}")
print(f"")
print("--- Head node logs (last 2000 chars) ---")
print(job.get_logs()[-2000:])

## 8. Schedule HPO Pipeline via Task Graph (DAGTask)

Using **Snowflake Task Graphs** with `DAGTask(definition=...)` to schedule the ML Job.
This is the production-grade approach:
- No SQL `EXECUTE JOB SERVICE` needed
- The `@remote` definition IS the task definition
- Supports DAG dependencies (data prep → HPO → post-processing)
- Full observability via Task history and ML Job logs

In [ ]:
from snowflake.core import Root
from snowflake.core.task.dagv1 import DAG, DAGOperation, DAGTask
from datetime import timedelta

api_root = Root(session)
schema_ref = api_root.databases[DB].schemas[SCHEMA]

dag = DAG(
    "RRD_HPO_PIPELINE",
    schedule=timedelta(weeks=1),
    stage_location=f"@{DB}.{SCHEMA}.HPO_STAGE",
)

with dag:
    data_refresh = DAGTask(
        "DATA_REFRESH",
        definition=f"""ALTER VIEW {DB}.{SCHEMA}.{TABLE_NAME}_FLAT REFRESH""",
    )

    hpo_task = DAGTask(
        "HPO_TRAINING",
        definition=hpo_distributed_job,
    )

    data_refresh >> hpo_task

print("DAG Definition:")
print(f"  Name:     RRD_HPO_PIPELINE")
print(f"  Schedule: Weekly")
print(f"  Tasks:    DATA_REFRESH -> HPO_TRAINING")
print(f"")
print("Deploy with:")
print(f"  DAGOperation(schema_ref).deploy(dag)")
print(f"")
print("Run manually:")
print(f"  DAGOperation(schema_ref).run(dag)")
print(f"")
print("The HPO_TRAINING task uses the @remote job definition directly.")
print("It provisions a 4-node GPU cluster, runs the full Tuner HPO,")
print("and registers the best model to the Model Registry.")

## 9. Summary & Key Findings for RR Donnelley

### Architecture: Tasks + ML Jobs for Production HPO

```
┌─────────────────────────────────────────────────────────────┐
│  Snowflake Task Graph (DAG)                                 │
│                                                             │
│  ┌──────────────┐         ┌─────────────────────────────┐  │
│  │ DATA_REFRESH │ ──────> │ HPO_TRAINING (ML Job)       │  │
│  │  (SQL Task)  │         │  @remote(target_instances=4)│  │
│  └──────────────┘         │  ┌───────────────────────┐  │  │
│                            │  │ Tuner API             │  │  │
│                            │  │  - 12 trials          │  │  │
│                            │  │  - RandomSearch       │  │  │
│                            │  │  - Each trial uses    │  │  │
│                            │  │    distributed XGBoost│  │  │
│                            │  │    across 4 GPU nodes │  │  │
│                            │  └───────────────────────┘  │  │
│                            │  → Registers best model     │  │
│                            └─────────────────────────────┘  │
│  Schedule: Weekly (CRON)                                    │
└─────────────────────────────────────────────────────────────┘
```

### Answers to Client Questions:

| Question | Answer |
|----------|--------|
| **Tuner + Distributed class** | Full Tuner API works with distributed `XGBEstimator`. Each HPO trial runs distributed training across the full multi-node cluster. |
| **Distributed only in notebooks?** | **No.** Multi-Node ML Jobs (>= 1.9.2) support distributed training with `target_instances=N`. |
| **Multi-node not scaling** | In notebooks: call `scale_cluster(N)`. In ML Jobs: set `target_instances=N` — auto-provisioned. |
| **HPO scheduling** | `DAGTask(definition=hpo_job_func)` in a Task Graph. No SPCS YAML, no container builds. |
| **Production approach** | ML Jobs + Task Graph is the recommended pattern. Replaces EXECUTE JOB SERVICE. |

### Deployment Architecture for 147M x 4,200 Dataset:

| Stage | Approach |
|-------|----------|
| **Interactive Dev** | Notebook Container Runtime + `scale_cluster(4)` + Tuner API |
| **Production HPO** | Multi-Node ML Job (`target_instances=4`) with Tuner inside |
| **Scheduling** | Task Graph with `DAGTask(definition=...)` — weekly CRON |
| **Data Prep** | 4XL warehouse to flatten JSON columns into wide features |
| **Model** | XGBoost distributed (or LightGBM for sparse JSON features) |
| **Registry** | Best model auto-registered with metrics after each HPO run |
| **Monitoring** | `job.get_logs()`, `job.status`, Ray Dashboard |